<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "076"
iso_country = "Brazil"
year = 2025
gdp = 2.280e12

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Brazil': 

Number of trade records present in dataset: 55349
Number of partner countries present in dataset: 192
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$208158.0m
Total weight within the dataset: 183205.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,076,Brazil,842,USA,Import,Sea,20220,020220 | Meat; of bovine anim,0202,0202 | Meat of bovine anima,02,02 | Meat and edible meat,21112,"Meat of bovine animals, frozen",Agriculture,19,CMT - Bovine meat prods,51484,50923.0,51484,kg,3260.0,3260.0,10775.6012,9779.0,6076.387009,65476.723146,['Container'],['Container'],166.895992,166.384301,166.438882,166.438882,166.689382,166.689382,175.969177,330.700482,191.366688,179.007114,179.772669,184.909982,171.733108,324.415004,204.243668,186.463942,181.213206,191.343829,168.011873,361.932821,248.559284,222.998154,204.688333,200.353941,164.361343,376.403660,300.826334,277.551437,222.504568,216.479949,160.783880,392.607472,387.377869,377.918291,236.362929,228.371298,157.136506,386.093597,371.118750,404.433279,317.224034,232.749910,153.468498,329.239847,356.727395,401.788690,300.241993,235.540339,149.973207,293.793973,300.098601,359.636824,287.372385,237.426404,21112
1,1,076,Brazil,858,Uruguay,Import,Sea,20220,020220 | Meat; of bovine anim,0202,0202 | Meat of bovine anima,02,02 | Meat and edible meat,21112,"Meat of bovine animals, frozen",Agriculture,19,CMT - Bovine meat prods,10127,10008.0,10127,kg,762.0,762.0,2119.5811,1710.0,1062.544410,2252.149049,['Container'],['Container'],5.740584,5.722984,5.724861,5.724861,5.733478,5.733478,6.052667,11.374833,6.582283,6.157161,6.183493,6.360197,5.906963,11.158636,7.025202,6.413647,6.233042,6.581496,5.778966,12.449106,8.549489,7.670284,7.040496,6.891410,5.653402,12.946847,10.347276,9.546709,7.653307,7.446083,5.530351,13.504197,13.324318,12.998945,8.129981,7.855100,5.404895,13.280144,12.765067,13.910959,10.911294,8.005707,5.278730,11.324592,12.270059,13.819995,10.327177,8.101688,5.158505,10.105390,10.322245,12.370132,9.884512,8.166561,21112


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $10594.9m (5.1%), ... 

                             cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               1928506505  7.100587e+09      0.9      3.9
15 | Animal, vegetable or  1750264396  1.189925e+09      0.8      0.6
22 | Beverages, spirits a  1113748186  8.540548e+08      0.5      0.5
18 | Cocoa and cocoa prep   824633827  2.775377e+08      0.4      0.2
11 | Products of the mill   708449154  1.218480e+09      0.3      0.7
08 | Fruit and nuts, edib   587092355  3.529312e+08      0.3      0.2
03 | Fish and crustaceans   519409062  1.062410e+08      0.2      0.1
20 | Preparations of vege   486419111  3.628157e+08      0.2      0.2
23 | Food industries, res   454363025  3.724677e+08      0.2      0.2
04 | Dairy produce; birds   422858367  1.492134e+08      0.2      0.1
21 | Miscellaneous edible   402479691  1.026300e+08      0

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $7415.4m (3.6%), ... 

                    cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       3865186582  3.275038e+09      1.9      1.8
Food & Beverages  3512397808  1.965014e+09      1.7      1.1
Fishing             37833444  4.853715e+06      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $5994.5m (2.9%), ... 

                                 cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     2710098155  1.390007e+09      1.3      0.8
VOL - Vegetable oils and fats  1249786823  1.066346e+09      0.6      0.6
OCR - Crops n.e.c.              644940614  2.306286e+08      0.3      0.1
V_F - Vegetables, fruit, nuts   578236048  4.839527e+08      0.3      0.3
MIL - Dairy products            497698041  2.188074e+08      0.2      0.1
OAP - Animal products n.e.c.    154999178  4.513615e+07      0.1      0.0
CMT - Bovine meat prods          42854285  8.906022e+06      0.0      0.0
OMT - Meat products n.e.c.       42592201  1.146763e+07      0.0      0.0
OSD - Oil seeds                  26468453  2.775174e+07      0.0      0.0
PCR - Processed rice             20251466  3.517680e+07      0.0      0.0
PFB - Plant-based fibers         13772088  1.9

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 5.1% 
GTAP 2.9% 
EORA 3.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16124/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,076,Brazil,842,USA,Import,Sea,20220,020220 | Meat; of bovine anim,0202,0202 | Meat of bovine anima,02,02 | Meat and edible meat,21112,"Meat of bovine animals, frozen",Agriculture,19,CMT - Bovine meat prods,51484,50923.0,51484,kg,3260.0,3260.0,10775.6012,9779.0,6076.387009,65476.723146,['Container'],['Container'],166.895992,166.384301,166.438882,166.438882,166.689382,166.689382,175.969177,330.700482,191.366688,179.007114,179.772669,184.909982,171.733108,324.415004,204.243668,186.463942,181.213206,191.343829,168.011873,361.932821,248.559284,222.998154,204.688333,200.353941,164.361343,376.403660,300.826334,277.551437,222.504568,216.479949,160.783880,392.607472,387.377869,377.918291,236.362929,228.371298,157.136506,386.093597,371.118750,404.433279,317.224034,232.749910,153.468498,329.239847,356.727395,401.788690,300.241993,235.540339,149.973207,293.793973,300.098601,359.636824,287.372385,237.426404,21112,NaN
1,1,076,Brazil,858,Uruguay,Import,Sea,20220,020220 | Meat; of bovine anim,0202,0202 | Meat of bovine anima,02,02 | Meat and edible meat,21112,"Meat of bovine animals, frozen",Agriculture,19,CMT - Bovine meat prods,10127,10008.0,10127,kg,762.0,762.0,2119.5811,1710.0,1062.544410,2252.149049,['Container'],['Container'],5.740584,5.722984,5.724861,5.724861,5.733478,5.733478,6.052667,11.374833,6.582283,6.157161,6.183493,6.360197,5.906963,11.158636,7.025202,6.413647,6.233042,6.581496,5.778966,12.449106,8.549489,7.670284,7.040496,6.891410,5.653402,12.946847,10.347276,9.546709,7.653307,7.446083,5.530351,13.504197,13.324318,12.998945,8.129981,7.855100,5.404895,13.280144,12.765067,13.910959,10.911294,8.005707,5.278730,11.324592,12.270059,13.819995,10.327177,8.101688,5.158505,10.105390,10.322245,12.370132,9.884512,8.166561,21112,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $6123.3m of $208158.0m (2.9%)
Volume: 10169.8 of 183205.0 million kilograms (5.6%)
Value-to-Weight: 0.6 from 1.14 US$ per kg (+-47.0%)
Average distance to market (volume-weighted): 3013.4 from 7501.9 nm (+-59.8%)

            food_group  N_com    cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     26  1905870856  7.061269e+09      0.9      3.9
1    Processed Cereals    735  1499468899  1.498487e+09      0.7      0.8
2        Fats and Oils    236  1316268298  6.834735e+08      0.6      0.4
3               Fruits    147   524979772  3.250867e+08      0.3      0.2
4                Dairy     44   422848384  1.492098e+08      0.2      0.1
5           Vegetables    205   304893228  3.015005e+08      0.1      0.2
6      Sugar and Honey    157   140756729  1.445217e+08      0.1      0.1
7        Meat and Prep     18     8235561  6.234556e+06      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,9.088537e+07,1.701964e+08,9.877521e+07,9.241669e+07,9.280785e+07,9.547086e+07,8.310446e+07,2.018371e+08,1.990922e+08,1.944711e+08,1.218801e+08,1.177537e+08,7.756337e+07,1.513949e+08,1.546445e+08,1.851579e+08,1.480676e+08,1.224084e+08
1,Fats and Oils,2.176073e+09,3.576415e+09,2.316514e+09,2.176353e+09,2.183193e+09,2.260095e+09,2.038979e+09,4.054268e+09,3.967740e+09,4.083661e+09,2.786257e+09,2.652254e+09,1.942758e+09,3.331753e+09,3.395056e+09,3.935603e+09,3.256181e+09,2.734233e+09
2,Fruits,5.564116e+08,1.039504e+09,6.043604e+08,5.656058e+08,5.680301e+08,5.843510e+08,5.089964e+08,1.233931e+09,1.215835e+09,1.188826e+09,7.460297e+08,7.200101e+08,4.752527e+08,9.262756e+08,9.464809e+08,1.131635e+09,9.058264e+08,7.482644e+08
3,Meat and Prep,1.188666e+07,2.233872e+07,1.292676e+07,1.209188e+07,1.214359e+07,1.249061e+07,1.086090e+07,2.652051e+07,2.616726e+07,2.552826e+07,1.596624e+07,1.542641e+07,1.013064e+07,1.984569e+07,2.027157e+07,2.429336e+07,1.941192e+07,1.603808e+07
4,Processed Cereals,1.493814e+09,2.772022e+09,1.620240e+09,1.517118e+09,1.523675e+09,1.567805e+09,1.368331e+09,3.290134e+09,3.237843e+09,3.173575e+09,1.999495e+09,1.926533e+09,1.279071e+09,2.478750e+09,2.534132e+09,3.020986e+09,2.423801e+09,2.001106e+09
5,Sugar and Honey,2.978001e+08,5.458091e+08,3.222840e+08,3.019562e+08,3.032453e+08,3.121932e+08,2.734521e+08,6.461258e+08,6.349907e+08,6.252589e+08,3.969704e+08,3.817663e+08,2.561501e+08,4.905244e+08,5.015670e+08,5.956707e+08,4.795731e+08,3.962172e+08
6,Unprocessed Cereals,3.270099e+09,6.138482e+09,3.555373e+09,3.326038e+09,3.340285e+09,3.435869e+09,2.988582e+09,7.287368e+09,7.189005e+09,7.016150e+09,4.391228e+09,4.241462e+09,2.788183e+09,5.456718e+09,5.574316e+09,6.676786e+09,5.337299e+09,4.409237e+09
7,Vegetables,8.207090e+08,1.535739e+09,8.917228e+08,8.344214e+08,8.379984e+08,8.620500e+08,7.505318e+08,1.823114e+09,1.797042e+09,1.756065e+09,1.100962e+09,1.062847e+09,7.005839e+08,1.367324e+09,1.397045e+09,1.671367e+09,1.337291e+09,1.104669e+09


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.19,0.02,0.00,0.00,0.01,0.28,0.27,0.26,0.09,0.08,0.17,0.18,0.25,0.17,0.11
1,Fats and Oils,1.06,0.11,0.00,0.01,0.06,1.53,1.47,1.55,0.57,0.47,1.06,1.10,1.51,1.00,0.60
2,Fruits,0.92,0.09,0.02,0.02,0.05,1.38,1.35,1.29,0.45,0.40,0.86,0.90,1.25,0.82,0.52
3,Meat and Prep,1.27,0.13,0.02,0.03,0.07,1.90,1.86,1.78,0.62,0.55,1.18,1.23,1.72,1.13,0.72
4,Processed Cereals,0.85,0.08,0.02,0.02,0.05,1.28,1.25,1.20,0.42,0.37,0.80,0.84,1.16,0.76,0.48
5,Sugar and Honey,1.76,0.17,0.03,0.04,0.10,2.65,2.57,2.50,0.88,0.77,1.67,1.74,2.41,1.59,1.00
6,Unprocessed Cereals,1.51,0.15,0.03,0.04,0.09,2.26,2.20,2.11,0.74,0.66,1.40,1.46,2.04,1.34,0.85
7,Vegetables,2.35,0.23,0.04,0.06,0.14,3.52,3.43,3.30,1.15,1.02,2.19,2.28,3.18,2.09,1.33
